# Spark Streaming Notebook

This notebook is designed for the PostgreSQL + Debezium + Kafka stack in this repository. It shows how to consume change events from Kafka directly in a notebook and then process them with PySpark Structured Streaming.

The sample assumes:
- Docker Compose is running the PostgreSQL, Kafka, Debezium Connect, and Flink services.
- The Debezium PostgreSQL connector has been registered.
- The event generator is producing updates in the database.


## Configure Kafka and Debezium Source

The Debezium connector in this repository uses the topic prefix `dbserver1`. For the `book_references` table, the emitted topic is normally:

```text
dbserver1.public.book_references
```

You can verify the connector and topic from the REST API or with Kafka tooling.

In [1]:
import os

KAFKA_HOST = os.environ["KAFKA_HOST"]
KAFKA_PORT = os.environ["KAFKA_PORT"]

KAFKA_BOOTSTRAP_SERVERS = f"{KAFKA_HOST}:{KAFKA_PORT}"

SPARK_CONNECT_HOST = os.environ["SPARK_CONNECT_HOST"]
SPARK_CONNECT_PORT = os.environ["SPARK_CONNECT_PORT"]

SPARK_CONNECT_SERVER = f"{SPARK_CONNECT_HOST}:{SPARK_CONNECT_PORT}"

print(f"KAFKA_BOOTSTRAP_SERVERS: {KAFKA_BOOTSTRAP_SERVERS}")
print(f"SPARK_CONNECT_SERVER: {SPARK_CONNECT_SERVER}")

PG_DSN: dbname=monitoring user=monitoring password=mtg host=postgres port=5432
KAFKA_BOOTSTRAP_SERVERS: kafka-broker:29092
SPARK_CONNECT_SERVER: spark:15002


## Stream Process Events with Spark Structured Streaming

This section shows how to read the same topic from PySpark Structured Streaming, parse the Debezium JSON payload, and transform it. In practice, this is a good next step for learning streaming ETL and windowed aggregations.

In [2]:
from pyspark.sql import SparkSession

def kafka_topic_to_console(spark: SparkSession, topic: str):
    df = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
        .option("subscribe", topic) \
        .option("startingOffsets", "earliest") \
        .load()
    query = df.selectExpr("CAST(value AS STRING) as json") \
        .writeStream \
        .outputMode("append") \
        .format("console") \
        .option("truncate", False) \
        .queryName(topic) \
        .start()
    print(f"PySpark Kafka ({topic}) stream started successfully")
    return query

In [ ]:
import time

spark = SparkSession.builder \
    .appName("debezium-stream") \
    .remote(f"sc://{SPARK_CONNECT_SERVER}") \
    .getOrCreate()

try:
    references_query = kafka_topic_to_console(spark, "dbserver1.public.book_references")
    inventories_query = kafka_topic_to_console(spark, "dbserver1.public.book_inventories")
    rentals_query = kafka_topic_to_console(spark, "dbserver1.public.book_rentals")
    while True:
        time.sleep(10)
finally:
    spark.stop()

PySpark Kafka (dbserver1.public.book_references) stream started successfully
PySpark Kafka (dbserver1.public.book_inventories) stream started successfully
PySpark Kafka (dbserver1.public.book_rentals) stream started successfully
query               status       batch    in/s      proc/s    dur(ms)   age       error                         
------------------  -----------  -------  --------  --------  --------  --------  ------------------------------
03723ffa-6b8d-4134  TERMINATED   -        -         -         -         18s       -                             
0ad2701d-e182-4891  TERMINATED   -        -         -         -         121s      -                             
763d46b1-756b-49f5  TERMINATED   -        -         -         -         121s      -                             
8af5b43b-32d3-4d07  TERMINATED   -        -         -         -         18s       -                             
917c53fb-f6f0-4267  TERMINATED   -        -         -         -         121s      -          